#### LIBRARY IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import glob # For file pattern matching for loading files
import os # Certain debugging operations
import joblib # To save scaler and encoder

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Neural Netowrk specifc imports
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

#### CONFIGURATOINS

In [ ]:
FEATURE_ROOT = "eGeMAPs_features"
OUTPUT_ROOT = "neural-network"
RANDOM_SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

#### FUNCTIONS

In [ ]:
# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
def load_participant_data(participant_folder):
    csv_files = sorted(glob.glob(f"{FEATURE_ROOT}/{participant_folder}/*.csv")) # For multiple files

    # print(df.head())
    # print(df.shape)

    # Train / Validation / Test Split
    
    # # 80% train, 20% temp
    train_files, temp_files = train_test_split(
    csv_files,
    test_size= 1 - TRAIN_RATIO,
    random_state=RANDOM_SEED
    )

    # Split remaining into validation and test
    val_files, test_files = train_test_split(
    temp_files,
    test_size = TEST_RATIO/(VAL_RATIO+TEST_RATIO),
    random_state=RANDOM_SEED
    )

    print("Training Sessions")
    print(train_files)

    print("\nValidation Sessions")
    print(val_files)
  
    print("\nTesting Sessions")
    print(test_files)
    
    # Read and concatenate each split

    train_df = pd.concat(
        [pd.read_csv(f) for f in train_files],
        ignore_index=True
    )
    
    val_df = pd.concat(
        [pd.read_csv(f) for f in val_files],
        ignore_index=True
    )

    test_df = pd.concat(
        [pd.read_csv(f) for f in test_files],
        ignore_index=True
    )
    
    # Remove unnecessary columns
    train_df = train_df.drop(columns=["Unnamed: 0", "start_time", "end_time"])
    print(train_df.columns)
    val_df = val_df.drop(columns=["Unnamed: 0", "start_time", "end_time"])
    test_df = test_df.drop(columns=["Unnamed: 0", "start_time", "end_time"])

    # Seperate Features and Labels
    X_train = train_df.drop(columns=["label"])
    y_train = train_df["label"]

    X_val = val_df.drop(columns=["label"])
    y_val = val_df["label"]

    X_test = test_df.drop(columns=["label"])
    y_test = test_df["label"]

    # Print unique labels for debugging
    print(np.unique(y_train))
    print(np.unique(y_val))
    print(np.unique(y_test))
    
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
def preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test):
    
    # Label Encoding
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)
    y_val = encoder.transform(y_val)
    y_test = encoder.transform(y_test)

    # Print for debugging purpose
    print(encoder.classes_)
    print(np.unique(y_train))
    print(np.unique(y_val))
    print(np.unique(y_test))

    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train) # Learn the Meand and std and transform the data
    X_val = scaler.transform(X_val) # Transform the data using the learned mean and std
    X_test = scaler.transform(X_test) # Transform the data using the learned mean and std

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        scaler,
        encoder
    )  

In [ ]:
def build_model(input_dim):
    
    model = Sequential([
        Input(shape=(input_dim,)),

        # Hidden Layer 1
        Dense(128, activation='relu'),
        Dropout(0.3),

        # Hidden Layer 2
        Dense(64, activation='relu'),
        Dropout(0.3),

        # Output Layer 
        Dense(1, activation='sigmoid')
    ])

    # Compile Model
    model.compile(
        optimizer='adam', # Adam optimizer
        loss='binary_crossentropy', # Binary crossentropy loss
        metrics=['accuracy'] # Accuracy metric
    )
    
    #View the model
    model.summary()
    
    return model

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val):
    
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0       
    )
    
    return history

In [ ]:
def evaluate_model(
        model,
        X_test,
        y_test,
        encoder
    ):
    
    loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0   
    )
    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")

    probabilities = model.predict(X_test, verbose=0).flatten()
    predictions = (probabilities >= 0.5).astype(int)
    
    # Confusion MAtrix
    cm = confusion_matrix(y_test, predictions)
    
    metrics_dictionary = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1 Score": f1_score(y_test, predictions)
    }
    
    # Classification Report
    
    report = classification_report(
        y_test,
        predictions,
        target_names=encoder.classes_,
        output_dict=True
    )
    
    print(classification_report(
        y_test,
        predictions,
        target_names=encoder.classes_
    ))
    
    evaluation = {
        "metrics": metrics_dictionary,
        "predictions": predictions,
        "probabilities": probabilities,
        "actual": y_test,
        "confusion_matrix": cm,
        "classification_report": report
    }

    return evaluation

In [ ]:
def save_results(
        participant,
        model,
        scaler,
        encoder,
        history,
        evaluation
    ):
    
    metrics = evaluation["metrics"]
    predictions = evaluation["predictions"]
    probabilities = evaluation["probabilities"]
    actual = evaluation["actual"]
    cm = evaluation["confusion_matrix"]
    report = evaluation["classification_report"]
    
    # Save Model
    participant_output = Path(OUTPUT_ROOT, participant)
    os.makedirs(participant_output, exist_ok=True)
    model.save(Path(participant_output, "neural-network.keras"))
    
    # Save Scaler
    joblib.dump(scaler,Path(participant_output, "scaler.pkl"))
    
    # Save Encoder
    joblib.dump(encoder,Path(participant_output, "encoder.pkl"))
    
    # Save History
    history_df = pd.DataFrame(history.history)
    history_df.to_csv(Path(participant_output, "history.csv"),index=False)
    
    # Save Metrics
    metrics_df = pd.DataFrame([metrics])

    metrics_df.to_csv(Path(participant_output, "metrics.csv"),index=False)
        
    # Save Confusion Matrix
    cm_df = pd.DataFrame(cm,index=encoder.classes_,columns=encoder.classes_)
    cm_df.to_csv(Path(participant_output, "confusion_matrix.csv"))
    
    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(Path(participant_output, "classification_report.csv"))
    
    # Save predictions
    prediction_df = pd.DataFrame({
        "Actual": actual,
        "Prediction": predictions,
        "Probability": probabilities
    })

    prediction_df.to_csv(Path(participant_output, "predictions.csv"),index=False)

#### RUN MODEL

In [ ]:
summary_results = []

participants = sorted(os.listdir(FEATURE_ROOT))

for participant in participants:

    print(f"Training {participant}")
    
    # Load data
    X_train, X_val, X_test, y_train, y_val, y_test = load_participant_data(participant)
    
    # Preprocess data
    X_train, X_val, X_test, y_train, y_val, y_test, scaler, encoder = preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test)
    
    # Build model
    model = build_model(X_train.shape[1])
    
    # Train model and return training history
    history = train_model(model, X_train, y_train, X_val, y_val)
    
    # Evaluate model and return evaluation metrics
    evaluation = evaluate_model(model, X_test, y_test, encoder)
    
    # Save results
    save_results(
        participant,
        model,
        scaler,
        encoder,
        history,
        evaluation
    )
    
    # Append results to summary
    summary_results.append({
        "Participant": participant,
        **evaluation["metrics"]
    })

# Save summary results as CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(
    Path(OUTPUT_ROOT,"summary_results.csv"),
    index=False
)
    